# 07 — Flashcard Generation

This notebook demonstrates the **Flashcard Generation Workflow** — a sequential pipeline that:
1. **Retrieves** relevant concepts from the knowledge base
2. **Generates** concise Q/A flashcards using an LLM
3. **Formats** output as validated Pydantic `Flashcard` models

Features:
- Concise answers (1-10 words)
- Hints for difficult cards
- Mnemonics where applicable
- Related topic linking
- Spaced repetition metadata (difficulty, source traceability)
- Difficulty-filtered generation

In [ ]:
import sys
sys.path.insert(0, '..')

from src.workflows.flashcards import FlashcardWorkflow
from src.llm import LLMClient
from src.retrieval import Retriever
from src.store import VectorStore, KnowledgeGraph
from models.output import Flashcard
from models.knowledge import Difficulty

## Setup

Initialize the knowledge store, retriever, and LLM client.

> **Note:** This assumes you have already run notebooks 01-04 to ingest documents,
> build the vector store, and populate the knowledge graph.

In [ ]:
# Initialize stores
vector_store = VectorStore(persist_directory="../chroma_db")
knowledge_graph = KnowledgeGraph()

# Initialize retriever and LLM
retriever = Retriever(vector_store=vector_store, knowledge_graph=knowledge_graph)
llm_client = LLMClient()

print(f"Available LLM providers: {llm_client.available_providers}")

## Create the Flashcard Workflow

In [ ]:
workflow = FlashcardWorkflow(retriever=retriever, llm_client=llm_client)
print("FlashcardWorkflow initialized.")

## Generate Flashcards

Generate a set of flashcards for a topic. The workflow automatically:
1. Retrieves relevant chunks from ChromaDB
2. Sends them to the LLM with a flashcard prompt
3. Validates output against the `Flashcard` Pydantic model

In [ ]:
# Generate 5 flashcards about a topic
topic = "photosynthesis"
cards = workflow.generate(topic=topic, num_cards=5)

print(f"Generated {len(cards)} flashcards for '{topic}'")
print(f"Type: {type(cards[0]).__name__}")

In [ ]:
# Display the generated flashcards
for i, card in enumerate(cards, 1):
    print(f"\n{'='*60}")
    print(f"Card {i} [{card.difficulty.value}]")
    print(f"{'='*60}")
    print(f"Q: {card.question}")
    print(f"A: {card.answer}")
    if card.hint:
        print(f"Hint: {card.hint}")
    if card.mnemonic:
        print(f"Mnemonic: {card.mnemonic}")
    if card.related_topics:
        print(f"Related: {', '.join(card.related_topics)}")
    print(f"Source chunks: {len(card.source_chunk_ids)}")

## Difficulty-Filtered Generation

Generate cards at a specific difficulty level. The retriever uses
metadata-filtered search when difficulty is specified.

In [ ]:
# Generate easy cards only
easy_cards = workflow.generate(topic="photosynthesis", difficulty="easy", num_cards=3)

print(f"Generated {len(easy_cards)} easy flashcards:")
for card in easy_cards:
    print(f"  [{card.difficulty.value}] Q: {card.question}")
    print(f"           A: {card.answer}")

## Conciseness Validation

Verify that answers are concise (1-10 words) — a key requirement
for effective flashcard study.

In [ ]:
# Check answer conciseness
print("Answer word counts:")
all_concise = True
for card in cards:
    word_count = len(card.answer.split())
    status = "✓" if 1 <= word_count <= 10 else "✗"
    if word_count > 10:
        all_concise = False
    print(f"  {status} ({word_count} words): {card.answer}")

print(f"\nAll answers concise (1-10 words): {all_concise}")

## Spaced Repetition Metadata

Each flashcard includes metadata useful for spaced repetition systems:
- `difficulty`: Used to set initial intervals
- `id`: Unique identifier for tracking review history
- `source_chunk_ids`: Traceability back to source material

In [ ]:
# Examine spaced repetition metadata
card = cards[0]
print("Spaced Repetition Metadata:")
print(f"  Card ID: {card.id}")
print(f"  Difficulty: {card.difficulty.value}")
print(f"  Source chunks: {card.source_chunk_ids}")
print(f"  Related topics: {card.related_topics}")
print()
print("Serialized JSON:")
print(card.to_json())

## Summary

The `FlashcardWorkflow` provides:
- **Retrieve → Generate → Format** pipeline
- Concise Q/A pairs (1-10 word answers)
- Hints and mnemonics for memorization
- Related topic linking for cross-reference
- Difficulty-aware generation with filtering
- Pydantic validation ensuring data integrity
- Source traceability via chunk IDs

Next steps:
- Integrate with spaced repetition scheduling (SM-2 algorithm)
- Feed into progress tracking for adaptive difficulty